# Jun-2026 LAPD ion-saturation current (Isat) — fluctuation explorer

Companion to `Jun2026_IV_explore.ipynb`.  That notebook walks the *swept*
Langmuir tips (complete I+V pairs) to extract Vp/Te/ne; this one reads a
**fixed-bias ion-saturation channel** and looks at how the Isat signal
*fluctuates* in time (raw trace + FFT).

Reader + FFT helpers live in `Jun2026_Isat.py` (which reuses the read path and
knobs from `Jun2026_IV.py`); this notebook just drives them.  **You pick the
Isat channel by hand** — the notebook prints every channel's description and the
run's probe settings, then you name the scope + channel.

**This stage saves nothing** — no files, no batch loop.

Workflow: configure → list channel descriptions + run setup → select channel →
positions → read raw signal + plot. *(Pause here — FFT / fluctuation analysis
comes next.)*

## 1. Configure — set the run file

In [ ]:
import importlib
get_ipython().run_line_magic("matplotlib", "qt")
# get_ipython().run_line_magic("matplotlib", "inline")

import numpy as np
import matplotlib.pyplot as plt

import Jun2026_IV as jiv
import Jun2026_Isat as jis
importlib.reload(jiv)
importlib.reload(jis)   # re-run this cell after editing Jun2026_Isat.py / Jun2026_IV.py

from data_analysis.io import open_lapd

# >>> SET THIS to the run you want to inspect <<<
# (a run with fixed-bias Isat channels, e.g. the 30-/32- 'Isat, ...' runs)
ifn = r"D:\data\LAPD\jun2026-jia\30-He-1kG-bias40V-LP-p25p29-line_2026-06-13.hdf5"

# Current scaling knobs live in Jun2026_IV.py and are reused here (RESISTOR,
# Aprobe, I_SIGN).  For fluctuation work the *shape* of the spectrum matters,
# not the absolute scaling.
print(f"RESISTOR = {jiv.RESISTOR}, Aprobe = {jiv.Aprobe}, I_SIGN = {jiv.I_SIGN}")

assert ifn, "Set `ifn` to a run file first."
run = open_lapd(ifn)
print("backend:", run.backend)

## 2. List channel descriptions + run setup

Print every scope group's channel descriptions and the run's hand-written
description (plasma / bias / probe settings).  Read these to decide which scope +
channel is the Isat channel you want — the selection is made by hand in the next
cell, nothing is guessed here.

In [ ]:
channels = jis.list_all_channels(ifn)
print()
jis.print_run_description(ifn);

## 3. Select the Isat channel

From the lists above, set the scope group and channel of the Isat signal you
want to inspect.

In [ ]:
# >>> SET THESE to the Isat channel you want (from the lists above) <<<
scope_name = "scope"     # e.g. "scope" / "machscope" / "lpscope"
chan       = "C1"        # e.g. "C1"

desc = channels.get(scope_name, {}).get(chan, "(no description)")
print(f"Selected Isat channel: scope '{scope_name}' {chan}  ->  {desc!r}")
assert scope_name in channels, f"scope {scope_name!r} not in {list(channels)}"
assert chan in channels[scope_name], f"channel {chan!r} not in {list(channels[scope_name])}"

## 4. Positions — pick one to inspect

In [ ]:
# If the run has more than one probe drive (multiple motion groups), set
# motion_group_name to the one whose probe carries the Isat channel above
# (read_lp_positions lists the groups in its error). None works for single-drive runs.
motion_group_name = None    # e.g. "<Hermes>    p29_LP"

pos_array, xpos, ypos, npos, nshot = jiv.read_lp_positions(ifn, motion_group_name)

pos_index = 10     # <<< which position to inspect (0 .. npos-1)
print(f"\nInspecting position index {pos_index} of {npos} "
      f"(x={xpos[pos_index] if pos_index < len(xpos) else '?'} cm), {nshot} shots")

## 5. Read raw Isat signal at this position + plot

Per-shot Isat traces (scaled per the knobs) at the chosen position.  Inspect the
raw signal: the mean level is the ion-saturation current; the scatter about it
shot-to-shot and the wiggle within a shot are the fluctuations we'll FFT next.

In [ ]:
# uses scope_name / chan selected in section 3
tarr, Iarr = jis.get_isat_at_position(run, scope_name, chan, npos, nshot, pos_index)
print("Iarr:", Iarr.shape, " tarr:", tarr.shape)

fig, ax = plt.subplots(figsize=(11, 4))
for s in range(Iarr.shape[0]):
    ax.plot(tarr * 1e3, Iarr[s], lw=0.6, alpha=0.5)
ax.plot(tarr * 1e3, Iarr.mean(0), "k", lw=1.2, label="shot mean")
ax.set_xlabel("t [ms]"); ax.set_ylabel("Isat [a.u.]")
ax.set_title(f"Raw Isat — pos {pos_index} (x={xpos[pos_index] if pos_index < len(xpos) else '?'} cm), "
             f"scope '{scope_name}' {chan}: {desc!r}")
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

---
**Pause here.** Raw Isat signal read and plotted for one position/channel.
Next step: FFT the fluctuations (per-shot + averaged spectra) — to be added.